In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Major_Dhyan_Chand_National_Stadium_Delhi_DPCC_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,284.0,161.0,177.0,93.0,90.0,84.0,78.0,83.0,150.0,128.0,375.0,396.0
1,2,358.0,193.0,214.0,114.0,85.0,105.0,80.0,107.0,142.0,125.0,366.0,369.0
2,3,401.0,180.0,134.0,151.0,107.0,110.0,114.0,80.0,143.0,126.0,495.0,328.0
3,4,369.0,235.0,102.0,101.0,88.0,152.0,154.0,95.0,137.0,149.0,403.0,308.0
4,5,365.0,265.0,117.0,112.0,163.0,142.0,98.0,86.0,119.0,147.0,NaN,296.0
5,6,419.0,271.0,106.0,146.0,218.0,106.0,81.0,102.0,108.0,175.0,404.0,289.0
6,7,400.0,291.0,155.0,124.0,147.0,235.0,84.0,105.0,87.0,159.0,379.0,301.0
7,8,402.0,113.0,197.0,161.0,138.0,153.0,79.0,116.0,90.0,126.0,425.0,317.0
8,9,464.0,145.0,105.0,NaN,185.0,139.0,80.0,NaN,54.0,NaN,438.0,311.0
9,10,432.0,176.0,188.0,154.0,177.0,125.0,NaN,143.0,55.0,NaN,311.0,316.0


In [4]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   32 non-null     float64
 3   March      35 non-null     float64
 4   April      33 non-null     float64
 5   May        36 non-null     float64
 6   June       34 non-null     float64
 7   July       26 non-null     float64
 8   August     33 non-null     float64
 9   September  33 non-null     float64
 10  October    31 non-null     float64
 11  November   33 non-null     float64
 12  December   35 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


(41, 13)

In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,284.0,161.0,177.0,93.0,90.0,84.0,78.000000,83.0,150.0,128.0,375.000000,396.0
1,2,358.0,193.0,214.0,114.0,85.0,105.0,80.000000,107.0,142.0,125.0,366.000000,369.0
2,3,401.0,180.0,134.0,151.0,107.0,110.0,72.615385,80.0,143.0,126.0,495.000000,328.0
3,4,369.0,235.0,102.0,101.0,88.0,152.0,72.615385,95.0,137.0,149.0,403.000000,308.0
4,5,365.0,265.0,117.0,112.0,163.0,142.0,72.615385,86.0,119.0,147.0,314.848485,296.0


In [9]:
df_ml_ready.shape
df_ml_ready.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        31 non-null     int64  
 1   January    31 non-null     float64
 2   February   31 non-null     float64
 3   March      31 non-null     float64
 4   April      31 non-null     float64
 5   May        31 non-null     float64
 6   June       31 non-null     float64
 7   July       31 non-null     float64
 8   August     31 non-null     float64
 9   September  31 non-null     float64
 10  October    31 non-null     float64
 11  November   31 non-null     float64
 12  December   31 non-null     float64
dtypes: float64(12), int64(1)
memory usage: 3.3 KB
